# PHS 312 · L29/L30 In-Class Activity — Building a Hansch Equation

**Patrick 7e, Chapter 18** · Work in pairs · 30 minutes

Two published congeneric series: **coumarin MAO-B inhibitors** and **6,7-disubstituted tacrine
analogues**, each with measured activity and tabulated substituent constants. Everything runs in
the browser with no install.

| Part | Minutes | What you do |
| --- | --- | --- |
| A | 10 | One parameter at a time: is the relationship linear or parabolic? |
| B | 12 | Combine parameters into a Hansch equation and judge it honestly |
| C | 5  | Use the equation to pick the next compound to make |
| D | 3  | Report out |

**L30 pairs:** skip Part A and start at Part B.

In [ ]:
#@title Setup — run this first
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from itertools import combinations
np.set_printoptions(suppress=True)

def fit(X, y, names):
    """Least-squares fit with the statistics Patrick asks for: n, s, r."""
    X = np.column_stack([np.asarray(X, float).reshape(len(y), -1), np.ones(len(y))])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ beta
    resid = y - pred
    n, p = len(y), X.shape[1]
    ss_res, ss_tot = (resid**2).sum(), ((y - y.mean())**2).sum()
    r2 = 1 - ss_res/ss_tot
    s = np.sqrt(ss_res/(n - p)) if n > p else np.nan
    terms = " ".join(f"{b:+.3f}({nm})" for b, nm in zip(beta[:-1], names))
    eq = f"log(1/C) = {terms} {beta[-1]:+.3f}"
    return dict(eq=eq, beta=beta, r=np.sqrt(max(r2,0)), r2=r2, s=s, n=n, pred=pred)

def report(res):
    print(res["eq"])
    print(f"   n = {res['n']}   r = {res['r']:.3f}   r^2 = {res['r2']:.3f}   s = {res['s']:.3f}   "
          f"100r^2 = {100*res['r2']:.0f}% of the variation explained")
print("Ready.")

## Part A — Exercise 18.2: coumarin MAO-B inhibitors

Eleven coumarins differing only in the substituent X. For each you have the measured activity and
three substituent parameters.

In [ ]:
ex182 = pd.DataFrame({
 "X":   ["H","Me","OH","OMe","OCF3","NH2","F","Cl","CF3","CN","NO2"],
 "pIC50":[8.36,8.38,8.01,8.44,7.94,7.46,8.55,8.48,8.24,7.97,8.59],
 "pi":  [0.00,0.49,-0.67,-0.09,1.02,-1.23,0.14,0.71,0.88,-0.57,-0.26],
 "Vdw": [0.08,1.01,0.49,1.49,1.60,0.67,0.36,1.07,1.11,1.14,1.20],
 "F":   [0.00,-0.04,0.28,0.25,0.37,0.02,0.42,0.40,0.37,0.50,0.66]})
ex182

In [ ]:
y = ex182["pIC50"].values
fig, axes = plt.subplots(1, 3, figsize=(14,4))
for ax, col, lab in zip(axes, ["pi","Vdw","F"],
                        ["hydrophobicity  pi", "van der Waals volume", "inductive factor  F"]):
    x = ex182[col].values
    ax.scatter(x, y)
    for xi, yi, t in zip(x, y, ex182["X"]):
        ax.annotate(t, (xi, yi), fontsize=8, xytext=(3,3), textcoords="offset points")
    lin = fit(x, y, [col]); quad = fit(np.column_stack([x, x**2]), y, [col, col+"^2"])
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, lin["beta"][0]*xs + lin["beta"][1], "--",
            label=f"linear  r={lin['r']:.2f}")
    ax.plot(xs, quad["beta"][0]*xs + quad["beta"][1]*xs**2 + quad["beta"][2], "-",
            label=f"quadratic  r={quad['r']:.2f}")
    ax.set_xlabel(lab); ax.set_ylabel("pIC50"); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

for col in ["pi","Vdw","F"]:
    x = ex182[col].values
    print(f"--- {col}"); report(fit(x, y, [col]))
    report(fit(np.column_stack([x, x**2]), y, [col, col+"^2"])); print()

**A1.** Which single parameter correlates best with activity?

**A2.** For that parameter, is the fit better as a straight line or as a parabola? Quote both r values.

**A3.** Patrick warns that r always rises when you add a term. Look at **s** as well as r — does the
parabola actually explain more, or just fit more?

**A4.** NH2 is the least active compound. Which parameter explains that best?

## Part B — Exercise 18.4: a real multi-parameter Hansch equation

Tacrine and eleven 6,7-disubstituted analogues. Four parameter types, each for the X and the Y
position: hydrophobicity (pi), molar refractivity (MR), inductive (F) and resonance (R).

**Note on the data:** some printed tables give pi(Me) = 5.60. That is a typo for the standard
value **0.56**, which is what is used below — always sanity-check a parameter table before you
trust a regression built on it.

In [ ]:
pi = {"H":0.00,"Me":0.56,"Cl":0.71,"NO2":-0.28,"OMe":-0.02,"NH2":-1.23,"F":0.14}
MR = {"H":0.103,"Me":0.565,"Cl":0.603,"NO2":0.736,"OMe":0.787,"NH2":0.542,"F":0.092}
Fc = {"H":0.00,"Me":0.01,"Cl":0.42,"NO2":0.65,"OMe":0.29,"NH2":0.08,"F":0.45}
Rc = {"H":0.00,"Me":-0.18,"Cl":-0.19,"NO2":0.13,"OMe":-0.56,"NH2":-0.74,"F":-0.39}

rows = [("tacrine","H","H",6.59), ("I","Me","H",5.09), ("II","H","Me",7.00),
        ("III","Cl","H",6.26), ("IV","H","Cl",8.00), ("V","NO2","H",5.52),
        ("VI","H","NO2",7.55), ("VII","H","OMe",6.45), ("VIII","NH2","H",5.42),
        ("IX","H","F",7.06), ("X","Cl","Cl",6.32), ("XI","OMe","OMe",5.29)]
ex184 = pd.DataFrame(rows, columns=["cpd","X","Y","pIC50"])
ex184["piX"]=ex184.X.map(pi); ex184["piY"]=ex184.Y.map(pi)
ex184["MRX"]=ex184.X.map(MR); ex184["MRY"]=ex184.Y.map(MR)
ex184["F_XY"]=ex184.X.map(Fc)+ex184.Y.map(Fc)
ex184["R_XY"]=ex184.X.map(Rc)+ex184.Y.map(Rc)
ex184

In [ ]:
y4 = ex184["pIC50"].values
params = ["piX","piY","MRX","MRY","F_XY","R_XY"]

print("=== one parameter at a time ===")
for p in params:
    res = fit(ex184[p].values, y4, [p]); print(f"{p:>5}: r = {res['r']:.3f}   s = {res['s']:.3f}")

print("\n=== best two-parameter combinations ===")
scored = []
for a, b in combinations(params, 2):
    res = fit(ex184[[a,b]].values, y4, [a,b]); scored.append((res["r"], a, b, res))
for r, a, b, res in sorted(scored, reverse=True)[:4]:
    report(res); print()

In [ ]:
#@title Try your own combination — edit the list and re-run
chosen = ["MRX","piY"]          # <-- change these and re-run
res = fit(ex184[chosen].values, y4, chosen)
report(res)

plt.figure(figsize=(5.2,5))
plt.scatter(res["pred"], y4)
for p_, o_, t in zip(res["pred"], y4, ex184["cpd"]):
    plt.annotate(t, (p_, o_), fontsize=8, xytext=(3,3), textcoords="offset points")
lims = [min(y4.min(), res["pred"].min())-.3, max(y4.max(), res["pred"].max())+.3]
plt.plot(lims, lims, "k--", lw=1); plt.xlim(lims); plt.ylim(lims)
plt.xlabel("predicted pIC50"); plt.ylabel("observed pIC50")
plt.title(" + ".join(chosen)); plt.grid(alpha=.3); plt.show()

**B1.** Write out your best two-parameter equation with n, s and r.

**B2.** Does a substituent at **X** or at **Y** matter more for activity? What does that say about
the shape of the binding site?

**B3.** Patrick's rule of thumb is r above 0.9 with a *reasonable number of results*. You have 12
compounds. How many parameters can you justify fitting, and why is adding a sixth a bad idea?

**B4.** Which compound does your equation predict worst? Look it up in the table — what is
unusual about it?

## Part C — Use the equation

A chemist can make one more analogue this month.

In [ ]:
#@title Predict a new analogue
X_new, Y_new = "H", "Cl"        # <-- choose substituents from the tables above

row = {"piX":pi[X_new], "piY":pi[Y_new], "MRX":MR[X_new], "MRY":MR[Y_new],
       "F_XY":Fc[X_new]+Fc[Y_new], "R_XY":Rc[X_new]+Rc[Y_new]}
x_new = np.array([[row[c] for c in chosen]])
pred = float((np.column_stack([x_new, [[1.0]]]) @ res["beta"]).ravel()[0])
print(f"X = {X_new}, Y = {Y_new}   ->  predicted pIC50 = {pred:.2f}")
print(f"(most active compound measured: {y4.max():.2f})")

**C1.** Which X/Y pair does your equation say is best? Is it one already in the table?

**C2.** Patrick's chapter ends with a CoMFA study of these same tacrine analogues that predicted
the 6-bromo analogue at pIC50 7.40, against 7.18 measured. Your equation has no bromine value in
its tables. What would you have to do to predict it, and what does that say about the limits of a
substituent-constant QSAR?

## Part D — Report out

1. Your best equation, read aloud with n, s and r.
2. The one compound you would make next, and why.

---
*Built for PHS 312, Fall 2026 (Dr. Bolden). Activity data are published pIC50 values for two
congeneric series — coumarin MAO-B inhibitors and 6,7-disubstituted tacrine analogues — used
with standard Hansch substituent constants (π, MR, F, R). Students following the course text
will recognise both series from Chapter 18.*